### Setup

In [1]:
# Library
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="1"
import torch
import json
from metric import * 
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore') 

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)   

# CONFIG
TEST_SIZE = 200

# PATH
CONFIG_PATH = '../config.json'
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
    
## DATA
SEED_IMAGE_FOLDER = config.get('SEED_IMAGE_FOLDER')
SEED_LABEL_FOLDER = config.get('SEED_LABEL_FOLDER')
SEED_LABEL_FILE = sorted(os.listdir(SEED_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
AUGEMNT_IMAGE_FOLDER = config.get('AUGEMNT_IMAGE_FOLDER')
AUGEMNT_LABEL_FOLDER =  config.get('AUGEMNT_LABEL_FOLDER')
AUGMENT_LABEL_FILE = sorted(os.listdir(AUGEMNT_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
ANSWER_INSTRUCTION_FOLDER = config.get('ANSWER_INSTRUCTION_FOLDER')
ANSWER_INSTRUCTION_FILE = sorted(os.listdir(ANSWER_INSTRUCTION_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
ANSWER_IMAGE_FOLDER = config.get('ANSWER_IMAGE_FOLDER')
ANSWER_IMAGE_FILE = sorted(os.listdir(ANSWER_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SD_IMAGE_FOLDER = config.get('SD_IMAGE_FOLDER')
SD_IMAGE_FILE = sorted(os.listdir(SD_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
FIGMA_IMAGE_FOLDER = config.get('FIGMA_IMAGE_FOLDER')
FIGMA_IMAGE_FILE = sorted(os.listdir(FIGMA_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]

## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"
SAVE_WEIGHTS_PATH = '../Experiment/model_weights/FIGMA_weights_20250604_234758'

2025-06-10 14:20:18.205653: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-10 14:20:18.249757: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-10 14:20:18.792186: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Instructions for updating:
non-resource variables are not supported in the long term
device :  cuda


### Augment Instruction Performace

In [2]:
def augment_instruction_performance(image_folder, instruction_folder, instruction_file):
    clip_scores = []
    for filename in tqdm(instruction_file):
        if filename.endswith('.json'):
            instruction_path = os.path.join(instruction_folder , filename)
            image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

        with open(instruction_path, 'r') as f:
            prompt_data = json.load(f)
            prompt = prompt_data.get("Prompt", "")
            caption = prompt_data.get("Input", {}).get("caption", "")
            add_info = prompt_data.get("Add_Info", "")
            
            final_instruction = " ".join([prompt, caption, add_info])
            clip_score = calculate_clip_score(image_path, final_instruction)
            clip_scores.append(clip_score)
            
    average_clip_score = sum(clip_scores) / len(clip_scores)
    print(f"Average CLIP Score: {average_clip_score:.4f}")

- seed image - seed label 

In [4]:
augment_instruction_performance(SEED_IMAGE_FOLDER, SEED_LABEL_FOLDER, SEED_LABEL_FILE)

100%|██████████| 200/200 [04:59<00:00,  1.50s/it]

Average CLIP Score: 0.2962


- augment image - augment label

In [9]:
augment_instruction_performance(AUGEMNT_IMAGE_FOLDER, AUGEMNT_LABEL_FOLDER, AUGMENT_LABEL_FILE)

100%|██████████| 200/200 [04:56<00:00,  1.48s/it]

Average CLIP Score: 0.2946


### Prompt Following Performance for Generate Model

In [2]:
def prompt_following_performance(image_folder, instruction_folder, instruction_file):
    clip_scores = []
    for filename in tqdm(instruction_file):
        if filename.endswith('.json'):
            instruction_path = os.path.join(instruction_folder, filename)
            image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

        with open(instruction_path, 'r') as f:
            instruction_data = json.load(f)
            final_instruction = instruction_data['summary']
            clip_score = calculate_clip_score(image_path, final_instruction)
            clip_scores.append(clip_score)
            
    average_clip_score = sum(clip_scores) / len(clip_scores)
    print(f"Average CLIP Score: {average_clip_score:.4f}")

- stablediffusion image - instruction

In [ ]:
prompt_following_performance(SD_IMAGE_FOLDER, ANSWER_INSTRUCTION_FOLDER, ANSWER_INSTRUCTION_FILE)

- figma image - instruction

In [ ]:
prompt_following_performance(FIGMA_IMAGE_FOLDER, ANSWER_INSTRUCTION_FOLDER, ANSWER_INSTRUCTION_FILE)

### Image Following Performance for Generate Model

- stablediffusion image - answer image

In [ ]:
fid_score_sd = calculate_fid(ANSWER_IMAGE_FOLDER,ANSWER_IMAGE_FILE,SD_IMAGE_FOLDER,SD_IMAGE_FILE)
lpips_sd = average_lpips(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)
clip_similarity_sd = average_clip_similarity(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)

print(f"SD's Average FID Score: {fid_score_sd:.4f}")
print(f"SD's Average LPIPS Score: {lpips_sd:.4f}")
print(f"SD's Average CLIP Similarity: {clip_similarity_sd:.4f}")

- figma image - answer image

In [ ]:
fid_score_figma = calculate_fid(ANSWER_IMAGE_FOLDER,ANSWER_IMAGE_FILE,FIGMA_IMAGE_FOLDER,FIGMA_IMAGE_FILE)
lpips_figma = average_lpips(ANSWER_IMAGE_FOLDER, FIGMA_IMAGE_FOLDER)
clip_similarity_figma = average_clip_similarity(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)

print(f"FIGMA's Average FID Score: {fid_score_figma:.4f}")
print(f"FIGMA's Average LPIPS Score: {lpips_figma:.4f}")
print(f"FIGMA's Average CLIP Similarity: {clip_similarity_figma:.4f}")